In [1]:
# Use user library; no sudo needed
dir.create(Sys.getenv("R_LIBS_USER"), recursive = TRUE, showWarnings = FALSE)
.libPaths(unique(c(Sys.getenv("R_LIBS_USER"), .libPaths())))
if (!requireNamespace("pacman", quietly = TRUE)) install.packages("pacman", repos="https://cloud.r-project.org")
pacman::p_load(msigdbr, dplyr, tibble, readr, jsonlite, stringr, purrr, tidyr)

# Find repo root (looks for configs/params.yaml)
repo_root <- function(start = getwd()){
  p <- normalizePath(start); for (i in 1:8){ if (file.exists(file.path(p,"configs","params.yaml"))) return(p); p <- dirname(p) }
  stop("configs/params.yaml not found")
}
BASE   <- repo_root()
MARKER <- file.path(BASE, "configs", "markers")
dir.create(MARKER, recursive = TRUE, showWarnings = FALSE)

sessionInfo()  # capture versions for reproducibility


R version 4.5.1 (2025-06-13)
Platform: x86_64-pc-linux-gnu
Running under: Ubuntu 22.04.5 LTS

Matrix products: default
BLAS:   /usr/lib/x86_64-linux-gnu/openblas-pthread/libblas.so.3 
LAPACK: /usr/lib/x86_64-linux-gnu/openblas-pthread/libopenblasp-r0.3.20.so;  LAPACK version 3.10.0

locale:
[1] C

time zone: Asia/Jerusalem
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] tidyr_1.3.1    purrr_1.0.2    stringr_1.5.1  jsonlite_1.8.8 readr_2.1.5   
[6] tibble_3.3.0   dplyr_1.1.4    msigdbr_25.1.1

loaded via a namespace (and not attached):
 [1] crayon_1.5.3      vctrs_0.6.5       cli_3.6.3         rlang_1.1.4      
 [5] stringi_1.8.4     generics_0.1.4    assertthat_0.2.1  glue_1.7.0       
 [9] htmltools_0.5.8.1 IRdisplay_1.1     IRkernel_1.3.2    hms_1.1.4        
[13] fansi_1.0.6       babelgene_22.9    evaluate_0.24.0   tzdb_0.5.0       
[17] fastmap_1.2.0     base64enc_0.1-3  

In [2]:
write_set <- function(df, gs_name, out_basename){
  genes <- df %>% filter(gs_name == !!gs_name) %>%
    pull(gene_symbol) %>% unique() %>% sort()
  out_txt  <- file.path(MARKER, paste0(out_basename, ".txt"))
  out_json <- file.path(MARKER, paste0(out_basename, ".json"))
  readr::write_lines(genes, out_txt)
  jsonlite::write_json(
    list(name = gs_name, genes = genes,
         msigdbr_version = as.character(packageVersion("msigdbr")),
         timestamp = as.character(Sys.time())),
    out_json, pretty = TRUE, auto_unbox = TRUE
  )
  message(sprintf("[saved] %s (%d genes)", out_txt, length(genes)))
}


In [3]:
suppressPackageStartupMessages({
  library(dplyr)
  library(msigdbr)
})

# 0) sanity — list collections for the mouse DB (note 'MH' = Hallmark)
msigdbr_collections(db_species = "MM")

# 1) pull mouse-native Hallmark sets (collection 'MH')
hall_mm <- msigdbr(
  species    = "Mus musculus",
  db_species = "MM",     # use the mouse MSigDB database itself
  collection = "MH"      # <-- Hallmark for the mouse DB
)

# quick checks (should say "Mus musculus" and set names start with HALLMARK_)
stopifnot(all(hall_mm$species_name == "Mus musculus"))
print(unique(hall_mm$gs_name)[1:10])

# 2) helper to write one set per file
write_set <- function(gs, path) {
  genes <- hall_mm %>%
    filter(gs_name == gs) %>%
    distinct(gene_symbol) %>%
    pull(gene_symbol)
  dir.create(dirname(path), recursive = TRUE, showWarnings = FALSE)
  writeLines(genes, path)
  message("[saved] ", path, " (", length(genes), " genes)")
}

DIR <- "configs/markers"

# 3) core sets for this project
write_set("HALLMARK_E2F_TARGETS",                        file.path(DIR, "S_mouse_msigdb.txt"))
write_set("HALLMARK_G2M_CHECKPOINT",                     file.path(DIR, "G2M_mouse_msigdb.txt"))
write_set("HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION",  file.path(DIR, "EMT_mouse_msigdb.txt"))
write_set("HALLMARK_TGF_BETA_SIGNALING",                 file.path(DIR, "tgf_beta_signaling_mouse_msigdb.txt"))
write_set("HALLMARK_HYPOXIA",                            file.path(DIR, "hypoxia_mouse_msigdb.txt"))
write_set("HALLMARK_INTERFERON_GAMMA_RESPONSE",          file.path(DIR, "interferon_gamma_response_mouse_msigdb.txt"))
write_set("HALLMARK_OXIDATIVE_PHOSPHORYLATION",          file.path(DIR, "oxidative_phosphorylation_mouse_msigdb.txt"))
write_set("HALLMARK_GLYCOLYSIS",                         file.path(DIR, "glycolysis_mouse_msigdb.txt"))
write_set("HALLMARK_MYC_TARGETS_V1",                     file.path(DIR, "myc_targets_v1_mouse_msigdb.txt"))
write_set("HALLMARK_UNFOLDED_PROTEIN_RESPONSE",          file.path(DIR, "unfolded_protein_response_mouse_msigdb.txt"))
write_set("HALLMARK_MTORC1_SIGNALING",                   file.path(DIR, "mtorc1_signaling_mouse_msigdb.txt"))
write_set("HALLMARK_DNA_REPAIR",                         file.path(DIR, "dna_repair_mouse_msigdb.txt"))


gs_collection,gs_subcollection,gs_collection_name,num_genesets
<chr>,<chr>,<chr>,<int>
M1,,Positional,341
M2,CGP,Chemical and Genetic Perturbations,980
M2,CP:BIOCARTA,BioCarta Pathways,252
M2,CP:REACTOME,Reactome Pathways,1309
M2,CP:WIKIPATHWAYS,WikiPathways,192
M3,GTRD,GTRD,279
M3,MIRDB,miRDB,1768
M5,GO:BP,GO Biological Process,7855
M5,GO:CC,GO Cellular Component,1040


Warning message:
"Unknown or uninitialised column: `species_name`."


 [1] "HALLMARK_ADIPOGENESIS"            "HALLMARK_ALLOGRAFT_REJECTION"    
 [3] "HALLMARK_ANDROGEN_RESPONSE"       "HALLMARK_ANGIOGENESIS"           
 [5] "HALLMARK_APICAL_JUNCTION"         "HALLMARK_APICAL_SURFACE"         
 [7] "HALLMARK_APOPTOSIS"               "HALLMARK_BILE_ACID_METABOLISM"   
 [9] "HALLMARK_CHOLESTEROL_HOMEOSTASIS" "HALLMARK_COAGULATION"            


[saved] configs/markers/S_mouse_msigdb.txt (199 genes)

[saved] configs/markers/G2M_mouse_msigdb.txt (195 genes)

[saved] configs/markers/EMT_mouse_msigdb.txt (194 genes)

[saved] configs/markers/tgf_beta_signaling_mouse_msigdb.txt (53 genes)

[saved] configs/markers/hypoxia_mouse_msigdb.txt (199 genes)

[saved] configs/markers/interferon_gamma_response_mouse_msigdb.txt (188 genes)

[saved] configs/markers/oxidative_phosphorylation_mouse_msigdb.txt (195 genes)

[saved] configs/markers/glycolysis_mouse_msigdb.txt (200 genes)

[saved] configs/markers/myc_targets_v1_mouse_msigdb.txt (197 genes)

[saved] configs/markers/unfolded_protein_response_mouse_msigdb.txt (111 genes)

[saved] configs/markers/mtorc1_signaling_mouse_msigdb.txt (199 genes)

[saved] configs/markers/dna_repair_mouse_msigdb.txt (148 genes)



In [4]:
# Hallmark names: should be the 50 HALLMARK_* sets
unique(hall_mm$gs_name) |> head()

# Spot-check that you’re getting mouse-style symbols (TitleCase)
hall_mm |>
  dplyr::filter(gs_name == "HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION") |>
  dplyr::slice_head(n = 12) |>
  dplyr::pull(gene_symbol)


[1] "HALLMARK_ADIPOGENESIS"        "HALLMARK_ALLOGRAFT_REJECTION"
[3] "HALLMARK_ANDROGEN_RESPONSE"   "HALLMARK_ANGIOGENESIS"       
[5] "HALLMARK_APICAL_JUNCTION"     "HALLMARK_APICAL_SURFACE"

[1] "Abi3bp" "Acta2"  "Adam12" "Anpep"  "Aplp1"  "Areg"   "Basp1"  "Bdnf"  
 [9] "Bgn"    "Bmp1"   "Cadm1"  "Cald1"

In [5]:
info <- list(
  msigdbr_version = as.character(utils::packageVersion("msigdbr")),
  msigdb_release  = unique(msigdbr::msigdbr_collections()$msigdb_version),
  date            = Sys.time(),
  species         = "Mus musculus",
  collections     = c("MH (Hallmark)", "M2:CP:REACTOME", "M3:GTRD", "MIRDB")
)
jsonlite::write_json(info, "configs/markers/_provenance_msigdbr.json", pretty=TRUE, auto_unbox=TRUE)


Warning message:
"Unknown or uninitialised column: `msigdb_version`."


In [6]:
write_gmt <- function(named_list, path) {
  con <- file(path, "w"); on.exit(close(con))
  for (nm in names(named_list)) {
    genes <- unique(named_list[[nm]])
    writeLines(paste(c(nm, "na", genes), collapse="\t"), con)
  }
}
# Build a named list from your saved files:
libdir <- "configs/markers"
markers <- list(
  HALLMARK_E2F_TARGETS          = readLines(file.path(libdir, "S_mouse_msigdb.txt")),
  HALLMARK_G2M_CHECKPOINT       = readLines(file.path(libdir, "G2M_mouse_msigdb.txt")),
  HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION = readLines(file.path(libdir, "EMT_mouse_msigdb.txt")),
  HALLMARK_TGF_BETA_SIGNALING   = readLines(file.path(libdir, "tgf_beta_signaling_mouse_msigdb.txt")),
  HALLMARK_HYPOXIA              = readLines(file.path(libdir, "hypoxia_mouse_msigdb.txt")),
  HALLMARK_INTERFERON_GAMMA_RESPONSE = readLines(file.path(libdir, "interferon_gamma_response_mouse_msigdb.txt")),
  HALLMARK_OXIDATIVE_PHOSPHORYLATION = readLines(file.path(libdir, "oxidative_phosphorylation_mouse_msigdb.txt")),
  HALLMARK_GLYCOLYSIS           = readLines(file.path(libdir, "glycolysis_mouse_msigdb.txt")),
  HALLMARK_MYC_TARGETS_V1       = readLines(file.path(libdir, "myc_targets_v1_mouse_msigdb.txt")),
  HALLMARK_UNFOLDED_PROTEIN_RESPONSE = readLines(file.path(libdir, "unfolded_protein_response_mouse_msigdb.txt")),
  HALLMARK_MTORC1_SIGNALING     = readLines(file.path(libdir, "mtorc1_signaling_mouse_msigdb.txt")),
  HALLMARK_DNA_REPAIR           = readLines(file.path(libdir, "dna_repair_mouse_msigdb.txt"))
)
write_gmt(markers, file.path(libdir, "hallmark_mouse_bundle.gmt"))


In [7]:
promoter_bed_dir <- "outputs/bed"
dir.create(promoter_bed_dir, showWarnings = FALSE, recursive = TRUE)

txdb <- TxDb.Mmusculus.UCSC.mm10.knownGene
gene2sym <- AnnotationDbi::select(org.Mm.eg.db, keys=keys(org.Mm.eg.db, "ENTREZID"),
                                  columns=c("SYMBOL"), keytype="ENTREZID") |> distinct()

# helper: SYMBOL -> promoters (GRanges)
promoters_from_symbols <- function(symbols, upstream=2000, downstream=2000) {
  # SYMBOL -> ENTREZ
  sym2ent <- AnnotationDbi::select(org.Mm.eg.db, keys=symbols,
                                   columns="ENTREZID", keytype="SYMBOL") |>
             drop_na(ENTREZID) |> distinct()
  # TSS per gene: take all transcripts and reduce to a single TSS range (strand-aware)
  tx <- transcripts(txdb, columns="gene_id")
  tx <- tx[tx$gene_id %in% sym2ent$ENTREZID]
  tss <- promoters(tx, upstream=upstream, downstream=downstream)
  # merge per gene (so you don’t double-count multi-transcript genes)
  reduce(tss)
}

# write one BED per marker file already saved in configs/markers
marker_dir <- "configs/markers"
marker_files <- list.files(marker_dir, pattern="\\.txt$", full.names=TRUE)

for (f in marker_files) {
  syms <- readLines(f); syms <- syms[nzchar(syms)]
  gr <- promoters_from_symbols(syms, 2000, 2000)
  out <- file.path(promoter_bed_dir, paste0("promoter2kb__", tools::file_path_sans_ext(basename(f)), ".bed"))
  rtracklayer::export(gr, out, format="BED")
  message(sprintf("[promoters] %s -> %s (%d ranges)", basename(f), out, length(gr)))
}


ERROR: Error in eval(expr, envir, enclos): object 'TxDb.Mmusculus.UCSC.mm10.knownGene' not found
